# PhysAR-IDS -- NB11 (v6): conformal prediction and supplementary statistics

**Questions.**
1. Does **split conformal prediction** give the IDS prediction sets with the promised coverage (1 - alpha) on the temporal test split, per class and overall, and what does it cost in set size?
2. What happens to coverage and set size **under attack** (PGD grey-box and tree-greedy white-box), and is the conformal score any use as an **adversarial signal**?
3. How certain are the headline clean numbers once the **burst structure** of the data is respected, and how well calibrated are the probabilities?

**Setup.** Tuned XGBoost (NB2 parameters), 5 seeds [42, 7, 123, 2024, 99]. Per (class, PLC) series the last 15 % of timestamps of the training split are the **calibration set**, preceded by a 2 % gap that is dropped; the conformal model is fitted on the remaining *proper-training* rows. The temporal test split is never touched for calibration. The full-train model (as in NB2) is fitted separately and is the one whose clean F1 is reported in section 8.

**Conformal methods** (alpha in 0.10, 0.05, 0.01): `lac` (score 1 - p_true, one marginal threshold), `lac_cc` (class-conditional / Mondrian threshold per true class), `aps` (randomised adaptive prediction sets), `lac_dedup` (LAC calibrated on one row per burst) and `lac_exch` (a control: calibrate on a random half of the calibration bursts, evaluate on the other half; this *is* exchangeable and must reach 1 - alpha if the code is right).

**Why bursts matter.** Rows sharing (class, PLC, timestamp) are identical copies, so the number of independent units is the number of bursts, not rows. All coverage intervals below are **burst-cluster bootstrap** intervals; the naive row-level half-width is printed next to them to show how much confidence the row count overstates.

**Limits stated up front.** The conformal guarantee needs calibration and test data to be exchangeable; the temporal split breaks that on purpose, so test coverage can fall below 1 - alpha and the size of the shortfall is a result, not a bug. The robust (NB5b) models were trained on the full training split including the calibration rows, so they cannot be calibrated honestly here and are not part of this notebook. Attacks are computed against the conformal model; the PGD examples come from the NB3 surrogate (transfer).

**Needs:** `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`, `surrogate_mlp_v6_seed{s}.pt` (NB3). **Runtime:** roughly 30-60 min on the local GPU.

## 0. Setup

In [ ]:
import os, sys, random, time, json, warnings, math
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
ALPHAS = [0.1, 0.05, 0.01]; ATK_ALPHAS = [0.1, 0.05]
CAL_FRAC = 0.15; GAP_FRAC = 0.02
N_BOOT = 2000
PGD_EPS = 0.1; PGD_ITERS = 10
WB_PER_CLASS = 300; WB_BUDGET = 0.3                  # white-box: sample per attack class, sigma on the 60 cyber features
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
N_BINS = 15

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"}')
print('Working directory:', os.getcwd())

## 1. Load v6 data, bursts and the calibration split

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']).astype(np.int64); y_test = np.asarray(data['y_test']).astype(np.int64)
plc_tr, plc_te = np.asarray(data['PLCID_train']), np.asarray(data['PLCID_test']); ts_tr, ts_te = np.asarray(data['timestamp_train']), np.asarray(data['timestamp_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']; atk_names = class_names[1:]
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
for k_ in ('n_estimators', 'max_depth', 'min_child_weight'):
    if k_ in best_params: best_params[k_] = int(best_params[k_])
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)

def burst_ids(y, plc, ts): return pd.DataFrame({'y': y, 'p': plc, 't': ts}).groupby(['y', 'p', 't'], sort=False).ngroup().to_numpy()
bid_tr, bid_te = burst_ids(y_train, plc_tr, ts_tr), burst_ids(y_test, plc_te, ts_te)

# calibration split: last CAL_FRAC of the distinct timestamps of every (class, PLC) series, a GAP_FRAC gap before it is dropped
_df = pd.DataFrame({'y': y_train, 'p': plc_tr, 't': ts_tr}); _g = _df.groupby(['y', 'p'])['t']
frac = ((_g.rank(method='dense') - 1) / _g.transform('nunique')).to_numpy()
cal_m = frac >= 1 - CAL_FRAC; gap_m = (frac >= 1 - CAL_FRAC - GAP_FRAC) & ~cal_m; prop_m = ~cal_m & ~gap_m
X_prop, y_prop = X_train[prop_m], y_train[prop_m]; X_cal, y_cal, bid_cal = X_train[cal_m], y_train[cal_m], bid_tr[cal_m]
atk_te = y_test != 0
print(f'features {N_FEAT} | train {len(y_train):,} = proper {prop_m.sum():,} + gap {gap_m.sum():,} + calibration {cal_m.sum():,} | test {len(y_test):,}')
print('calibration rows per class:', np.bincount(y_cal, minlength=6).tolist(), '| test rows per class:', np.bincount(y_test, minlength=6).tolist())
nb_cal = len(np.unique(bid_cal)); nb_te = len(np.unique(bid_te))
bc_cal = [len(np.unique(bid_cal[y_cal == c])) for c in range(6)]; bc_te = [len(np.unique(bid_te[y_test == c])) for c in range(6)]
print(f'independent units: calibration {nb_cal} bursts {bc_cal} (rows/burst median {np.median(np.bincount(np.unique(bid_cal, return_inverse=True)[1])):.0f}) | test {nb_te} bursts {bc_te} (rows/burst median {np.median(np.bincount(np.unique(bid_te, return_inverse=True)[1])):.0f})')
assert not set(np.unique(bid_cal)) & set(np.unique(bid_tr[prop_m])), 'a burst straddles proper-train and calibration'

## 2. Attacks (NB3c / NB5b white-box code, NB3 PGD) and model helpers

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def pgd(model, X, y, eps=PGD_EPS, n_iter=PGD_ITERS, bs=65536):
    model.eval(); alpha = eps / 4; out = np.empty_like(X)
    for s in range(0, len(X), bs):
        yt = torch.from_numpy(y[s:s + bs]).long().to(DEVICE); X0 = torch.from_numpy(X[s:s + bs]).float().to(DEVICE); Xa = X0.clone()
        for _ in range(n_iter):
            Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]; Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
        out[s:s + bs] = Xa.detach().cpu().numpy()
    return out
def fit_pair(X, y, seed):
    # GPU-fitted classifier (for the white-box margin search) and a CPU copy (the oracle for every reported number, as in NB4)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X, y, verbose=False)
    tmp = f'_tmp_nb11_{seed}_{os.getpid()}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu
print('helpers defined')

## 3. Conformal machinery

In [ ]:
def aps_scores(P, U):
    # randomised APS: score of label y = (mass of labels with higher probability) + U * p_y ; one U per row
    order = np.argsort(-P, 1); Ps = np.take_along_axis(P, order, 1); cum = np.cumsum(Ps, 1)
    S_sorted = cum - (1 - U)[:, None] * Ps; S = np.empty_like(P); np.put_along_axis(S, order, S_sorted, 1); return S
def scores_of(P, method, U): return aps_scores(P, U) if method == 'aps' else 1.0 - P
def qhat(s, alpha):
    n = len(s); k = int(math.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else float(np.sort(s)[k - 1])
def calibrate(Pc, yc, Uc, method, alpha, bid_c=None):
    # returns a scalar threshold, or a per-class vector for lac_cc; lac_dedup keeps one row per burst
    if method == 'lac_cc': return np.array([qhat(1 - Pc[yc == c, c], alpha) if (yc == c).any() else np.inf for c in range(6)])
    if method == 'lac_dedup':
        keep = np.unique(bid_c, return_index=True)[1]; return qhat(1 - Pc[keep, yc[keep]], alpha)
    S = scores_of(Pc, method, Uc); return qhat(S[np.arange(len(yc)), yc], alpha)
def predict_sets(P, method, q, U):
    S = scores_of(P, 'aps' if method == 'aps' else 'lac', U)
    return S <= (q[None, :] if np.ndim(q) else q)
def eval_sets(sets, y, bid, boot_seed=0):
    n = len(y); covered = sets[np.arange(n), y]; size = sets.sum(1); out = {}
    out['cov'] = float(covered.mean()); out['size'] = float(size.mean()); out['single'] = float((size == 1).mean()); out['empty'] = float((size == 0).mean())
    out['cov_pc'] = np.array([covered[y == c].mean() if (y == c).any() else np.nan for c in range(6)]); out['size_pc'] = np.array([size[y == c].mean() if (y == c).any() else np.nan for c in range(6)])
    out['worst_cov'] = float(np.nanmin(out['cov_pc'][np.bincount(y, minlength=6) > 0]))
    out['only_benign'] = float(((size == 1) & sets[:, 0]).mean()); out['has_benign'] = float(sets[:, 0].mean())
    u, inv = np.unique(bid, return_inverse=True); nb = len(u); cb = np.bincount(inv, covered.astype(float), nb); nr = np.bincount(inv, minlength=nb).astype(float)
    idx = np.random.RandomState(boot_seed).randint(nb, size=(N_BOOT, nb)); cs = cb[idx].sum(1) / nr[idx].sum(1)
    out['cov_lo'], out['cov_hi'] = [float(v) for v in np.percentile(cs, [2.5, 97.5])]; out['n_bursts'] = nb
    p = out['cov']; out['cov_row_hw'] = float(1.96 * math.sqrt(max(p * (1 - p), 1e-12) / n))
    return out
def exch_control(Pc, yc, bid_c, alpha, rng, reps=20):
    ub = np.unique(bid_c); covs = []
    for _ in range(reps):
        A = set(rng.permutation(ub)[:len(ub) // 2].tolist()); inA = np.fromiter((b in A for b in bid_c), bool, len(bid_c))
        q = qhat(1 - Pc[inA, yc[inA]], alpha); covs.append(float(((1 - Pc[~inA, yc[~inA]]) <= q).mean()))
    return float(np.mean(covs))
def conf_by_burst(y, pred, bid):
    u, inv = np.unique(bid, return_inverse=True); return np.bincount(inv * 36 + y * 6 + pred, minlength=len(u) * 36).reshape(len(u), 6, 6).astype(float)
def f1_from_conf(C):
    tp = np.einsum('...ii->...i', C); rs = C.sum(-1); cs = C.sum(-2)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where(rs + cs > 0, 2 * tp / (rs + cs), 0.0); rec = np.where(rs > 0, tp / rs, np.nan)
    sup = rs > 0; macro = (f1 * sup).sum(-1) / sup.sum(-1); weighted = (f1 * rs).sum(-1) / rs.sum(-1)
    return macro, weighted, f1, rec
def ece_of(P, y, bins=N_BINS):
    conf = P.max(1); acc = (P.argmax(1) == y); b = np.minimum((conf * bins).astype(int), bins - 1); e = 0.0; rows = []
    for k in range(bins):
        m = b == k
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean()); rows.append((k, int(m.sum()), float(conf[m].mean()), float(acc[m].mean())))
    return float(e), rows
def brier(P, y): return float(((P - np.eye(6)[y]) ** 2).sum(1).mean())
def nll(P, y): return float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1)).mean())
def temp_scale(P, T): L = np.log(np.clip(P, 1e-12, 1)) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def auroc(neg, pos):
    if len(neg) == 0 or len(pos) == 0: return float('nan')
    return float(roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[neg, pos]))
print('conformal machinery defined')

## 4. Per-seed run

For each seed: the conformal model (proper-train) and the full-train model are fitted; conformal sets are built on the clean test split; the same sets are then evaluated on the attack rows after PGD and after the white-box attack (sample of 300 rows per attack class), and the conformal score is tested as an adversarial signal. Results are cached per seed.

In [ ]:
rng0 = np.random.RandomState(0)
wb_sel = np.concatenate([rng0.choice(np.where(y_test == c)[0], size=min(60 if SMOKE_TEST else WB_PER_CLASS, int((y_test == c).sum())), replace=False) for c in range(1, 6)])
METHODS = ['lac', 'lac_cc', 'aps', 'lac_dedup']
RES = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'conformal_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([best_params, seed, ALPHAS, ATK_ALPHAS, CAL_FRAC, GAP_FRAC, N_BOOT, WB_PER_CLASS, WB_BUDGET, PGD_EPS, PGD_ITERS, K_CAND, MAX_ITERS, float(X_train[:5].sum()), int(cal_m.sum()), SMOKE_TEST, 'nb11-v1'], sort_keys=True, default=str)
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: RES[seed] = saved['res']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); r = {}; rng = np.random.RandomState(seed + 1100)
    cm_gpu, cm = fit_pair(X_prop, y_prop, seed)                                        # conformal model (proper-train)
    full_gpu, full = fit_pair(X_train, y_train, seed); del full_gpu                    # full-train model (NB2 setting)
    Pc, Pt = cm.predict_proba(X_cal), cm.predict_proba(X_test); Uc, Ut = rng.rand(len(y_cal)), rng.rand(len(y_test))
    Pfull = full.predict_proba(X_test)
    # ---- clean performance
    for nm, P in [('cm', Pt), ('full', Pfull)]:
        C = conf_by_burst(y_test, P.argmax(1), bid_te); mac, wei, f1c, rec = f1_from_conf(C.sum(0)); r[f'perf|{nm}|macro'] = float(mac); r[f'perf|{nm}|weighted'] = float(wei); r[f'perf|{nm}|f1_pc'] = f1c; r[f'perf|{nm}|recall_pc'] = rec
    # ---- bootstrap CIs of the full-train model: burst-cluster vs naive row-level
    C = conf_by_burst(y_test, Pfull.argmax(1), bid_te); nb = C.shape[0]; Cf = C.reshape(nb, 36); N = C.sum()
    Wb = rng.multinomial(nb, np.ones(nb) / nb, size=N_BOOT); Cb = (Wb @ Cf).reshape(-1, 6, 6)
    Cr = rng.multinomial(int(N), Cf.sum(0) / N, size=N_BOOT).reshape(-1, 6, 6)
    for tag, CC in [('burst', Cb), ('row', Cr)]:
        mac, wei, f1c, rec = f1_from_conf(CC)
        for nm, v in [('macro', mac), ('weighted', wei)]: r[f'ci|{tag}|{nm}'] = np.percentile(v, [2.5, 97.5])
        r[f'ci|{tag}|f1_pc'] = np.nanpercentile(f1c, [2.5, 97.5], axis=0); r[f'ci|{tag}|recall_pc'] = np.nanpercentile(rec, [2.5, 97.5], axis=0)
    # ---- calibration of the probabilities
    Tres = minimize_scalar(lambda T: nll(temp_scale(Pc, T), y_cal), bounds=(0.2, 5.0), method='bounded'); T_ = float(Tres.x); r['calib|T'] = T_
    for nm, P in [('cm', Pt), ('cm_T', temp_scale(Pt, T_)), ('full', Pfull)]:
        e, rows = ece_of(P, y_test); r[f'calib|{nm}|ece'] = e; r[f'calib|{nm}|brier'] = brier(P, y_test); r[f'calib|{nm}|nll'] = nll(P, y_test); r[f'calib|{nm}|bins'] = rows
    e, rows = ece_of(Pc, y_cal); r['calib|cal_set|ece'] = e
    # ---- conformal sets on the clean test split
    for a in ALPHAS:
        for m in METHODS:
            q = calibrate(Pc, y_cal, Uc, m, a, bid_cal); sets = predict_sets(Pt, m, q, Ut)
            for k, v in eval_sets(sets, y_test, bid_te).items(): r[f'clean|{m}|{a:g}|{k}'] = v
            r[f'clean|{m}|{a:g}|q'] = q
        r[f'clean|lac_exch|{a:g}|cov'] = exch_control(Pc, y_cal, bid_cal, a, rng)
    Q = {(m, a): calibrate(Pc, y_cal, Uc, m, a, bid_cal) for m in ('lac', 'aps') for a in ATK_ALPHAS}
    # ---- attacked rows
    sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE)); sur.eval()
    ia = np.flatnonzero(atk_te); X_pgd = pgd(sur, X_test[ia], y_test[ia]); P_pgd = cm.predict_proba(X_pgd)
    TH = get_thresholds(cm_gpu); mg = Margin(cm_gpu); X_wb = whitebox_trace(mg, X_test[wb_sel], y_test[wb_sel], CYBER_IDX, TH, WB_BUDGET, verbose=False)[0]; P_wb = cm.predict_proba(X_wb)
    groups = {'clean_atk': (Pt[ia], y_test[ia], bid_te[ia], Ut[ia]), 'pgd': (P_pgd, y_test[ia], bid_te[ia], Ut[ia]),
              'clean_wb': (Pt[wb_sel], y_test[wb_sel], bid_te[wb_sel], Ut[wb_sel]), 'wb': (P_wb, y_test[wb_sel], bid_te[wb_sel], Ut[wb_sel])}
    for gname, (P, y_, b_, U_) in groups.items():
        r[f'atk|{gname}|pt_evasion'] = float((P.argmax(1) == 0).mean()); r[f'atk|{gname}|pt_recall_pc'] = np.array([(P.argmax(1)[y_ == c] == c).mean() if (y_ == c).any() else np.nan for c in range(6)])
        r[f'atk|{gname}|pmax'] = float(P.max(1).mean())
        for m in ('lac', 'aps'):
            for a in ATK_ALPHAS:
                for k, v in eval_sets(predict_sets(P, m, Q[(m, a)], U_), y_, b_).items(): r[f'atk|{gname}|{m}|{a:g}|{k}'] = v
    # ---- the conformal score as an adversarial signal
    q01 = Q[('lac', 0.1)]; size_of = lambda P: predict_sets(P, 'lac', q01, np.zeros(len(P))).sum(1)
    benign_clean = Pt[y_test == 0]
    for gname, P_adv, P_cl in [('pgd', P_pgd, Pt[ia]), ('wb', P_wb, Pt[wb_sel])]:
        ev = P_adv.argmax(1) == 0
        for sname, f in [('1-pmax', lambda P: 1 - P.max(1)), ('setsize', size_of)]:
            r[f'det|{gname}|A|{sname}'] = auroc(f(P_cl), f(P_adv))                      # clean attack rows vs the same rows after the attack
            r[f'det|{gname}|B|{sname}'] = auroc(f(benign_clean), f(P_adv[ev])) if ev.any() else float('nan')   # genuine benign rows vs the rows that evaded
        r[f'det|{gname}|n_evaded'] = int(ev.sum())
    RES[seed] = r; joblib.dump({'fp': cfg_fp, 'res': r}, pf)
    print(f'  seed {seed} done in {time.time()-t0:.0f}s | clean F1 full {r["perf|full|weighted"]:.4f} (macro {r["perf|full|macro"]:.4f}) | LAC@0.1 cov {r["clean|lac|0.1|cov"]:.3f} size {r["clean|lac|0.1|size"]:.2f} | PGD cov {r["atk|pgd|lac|0.1|cov"]:.3f} | wb cov {r["atk|wb|lac|0.1|cov"]:.3f}', flush=True)
print(f'all seeds done in {(time.time()-t_all)/60:.1f} min')

## 5. Statistics helpers

In [ ]:
def ptest(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (float('nan') if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
def dz(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); s = d.std(ddof=1) if len(d) > 1 else 0
    return float(d.mean() / s) if s > 0 else float('nan')
def one_sample_p(x, mu):
    x = np.asarray(x, float)
    if len(x) < 2 or x.std(ddof=1) == 0: return float('nan')
    return float(stats.ttest_1samp(x, mu).pvalue)
def fp(p): return '< 0.0001' if (p == p and p < 5e-5) else (f'{p:.4f}' if p == p else 'n/a')
ms = lambda x, d=3: f'{np.mean(x):.{d}f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.{d}f}'
V = lambda key: np.array([RES[s][key] for s in SEEDS]); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); T = {}

## 6. Coverage and set size on the clean temporal test split

In [ ]:
print('6a. Marginal coverage and set size (mean +/- std over 5 seeds). CI = burst-cluster bootstrap 95 % interval (mean of the 5 seeds'' bounds); row hw = naive row-level half-width. p = coverage vs nominal 1 - alpha, one-sample t over seeds (Holm within this table)')
rows = []; pl = []
for a in ALPHAS:
    for m in METHODS:
        cov = V(f'clean|{m}|{a:g}|cov'); pl.append(one_sample_p(cov, 1 - a))
        rows.append({'alpha': a, 'method': m, 'nominal': 1 - a, 'coverage': ms(cov), 'burst CI 95%': f'[{V(f"clean|{m}|{a:g}|cov_lo").mean():.3f}, {V(f"clean|{m}|{a:g}|cov_hi").mean():.3f}]', 'row hw (naive)': f'+/-{V(f"clean|{m}|{a:g}|cov_row_hw").mean():.4f}',
                     'mean set size': ms(V(f'clean|{m}|{a:g}|size'), 2), 'singleton share': ms(V(f'clean|{m}|{a:g}|single')), 'empty share': ms(V(f'clean|{m}|{a:g}|empty')), 'worst-class coverage': ms(V(f'clean|{m}|{a:g}|worst_cov')), 'dz vs nominal': f'{dz(cov, np.full(len(cov), 1-a)):.2f}' if cov.std(ddof=1) > 0 else 'n/a'})
    cov = V(f'clean|lac_exch|{a:g}|cov'); pl.append(one_sample_p(cov, 1 - a))
    rows.append({'alpha': a, 'method': 'lac_exch (control)', 'nominal': 1 - a, 'coverage': ms(cov), 'burst CI 95%': '', 'row hw (naive)': '', 'mean set size': '', 'singleton share': '', 'empty share': '', 'worst-class coverage': '', 'dz vs nominal': ''})
ph = holm(pl)
for i, rw in enumerate(rows): rw['p (Holm)'] = fp(ph[i])
T['coverage'] = pd.DataFrame(rows); print(T['coverage'].to_string(index=False))
print('\n   Reading: coverage below 1 - alpha with a burst CI that excludes it is temporal-shift undercoverage. lac_exch (random split of the calibration bursts) should sit at about 1 - alpha (within a point or so); if it is clearly below, the implementation is wrong.')

print('\n6b. Per-class coverage (mean over seeds) and mean set size per true class')
rows = []
for a in ATK_ALPHAS:
    for m in ('lac', 'lac_cc', 'aps'):
        rows.append({'alpha': a, 'method': m, 'what': 'coverage', **{c: ms(V(f'clean|{m}|{a:g}|cov_pc')[:, i]) for i, c in enumerate(class_names)}})
        rows.append({'alpha': a, 'method': m, 'what': 'set size', **{c: ms(V(f'clean|{m}|{a:g}|size_pc')[:, i], 2) for i, c in enumerate(class_names)}})
T['perclass'] = pd.DataFrame(rows); print(T['perclass'].to_string(index=False))
print('   independent test bursts per class:', dict(zip(class_names, bc_te)), '-- per-class coverage is resolved only to roughly 1 / sqrt(bursts)')

print('\n6c. Method comparisons, paired over seeds (Holm within this table); dz = paired effect size')
rows = []; pl = []
for a in ALPHAS:
    for nm, f, k, A, B in [('set size: aps - lac', None, 'size', 'aps', 'lac'), ('worst-class coverage: lac_cc - lac', None, 'worst_cov', 'lac_cc', 'lac'), ('set size: lac_cc - lac', None, 'size', 'lac_cc', 'lac'), ('coverage: lac_dedup - lac', None, 'cov', 'lac_dedup', 'lac')]:
        x, y = V(f'clean|{A}|{a:g}|{k}'), V(f'clean|{B}|{a:g}|{k}'); pl.append(ptest(x, y))
        rows.append({'alpha': a, 'comparison': nm, 'mean difference': f'{(x - y).mean():+.4f}', 'dz': f'{dz(x, y):.2f}'})
ph = holm(pl)
for i, rw in enumerate(rows): rw['p (Holm)'] = fp(ph[i])
T['method_compare'] = pd.DataFrame(rows); print(T['method_compare'].to_string(index=False))

## 7. Under attack, and the conformal score as an adversarial signal

In [ ]:
print('7a. Conformal sets (calibrated on clean data) on attack rows: clean vs attacked. PGD = all test attack rows (grey-box transfer, 0.1 sigma); wb = tree-greedy white-box, 0.3 sigma on cyber features, 300 rows per attack class.')
print('    only {benign} = the set contains nothing but benign (the IDS stays silent); has benign = benign is in the set; pt evasion = arg-max is benign. p = attacked vs its own clean baseline (paired t over seeds, Holm within this table).')
rows = []; pl = []
for gname, base in [('pgd', 'clean_atk'), ('wb', 'clean_wb')]:
    for m in ('lac', 'aps'):
        for a in ATK_ALPHAS:
            for g_ in (base, gname):
                pass
            cb_, ca_ = V(f'atk|{base}|{m}|{a:g}|cov'), V(f'atk|{gname}|{m}|{a:g}|cov'); pl.append(ptest(ca_, cb_))
            rows.append({'attack': gname, 'method': m, 'alpha': a, 'coverage clean': ms(cb_), 'coverage attacked': ms(ca_), 'burst CI attacked': f'[{V(f"atk|{gname}|{m}|{a:g}|cov_lo").mean():.3f}, {V(f"atk|{gname}|{m}|{a:g}|cov_hi").mean():.3f}]', 'dz': f'{dz(ca_, cb_):.2f}',
                         'set size clean': ms(V(f'atk|{base}|{m}|{a:g}|size'), 2), 'set size attacked': ms(V(f'atk|{gname}|{m}|{a:g}|size'), 2),
                         'only {benign} clean': ms(V(f'atk|{base}|{m}|{a:g}|only_benign')), 'only {benign} attacked': ms(V(f'atk|{gname}|{m}|{a:g}|only_benign')), 'has benign attacked': ms(V(f'atk|{gname}|{m}|{a:g}|has_benign')),
                         'pt evasion clean': ms(V(f'atk|{base}|pt_evasion')), 'pt evasion attacked': ms(V(f'atk|{gname}|pt_evasion'))})
ph = holm(pl)
for i, rw in enumerate(rows): rw['p cov (Holm)'] = fp(ph[i])
T['attack'] = pd.DataFrame(rows); print(T['attack'].to_string(index=False))
print('\n7b. Per-class point recall on attack rows, clean vs attacked (mean over seeds)')
rows = [{'group': g, **{c: ms(V(f'atk|{g}|pt_recall_pc')[:, i]) for i, c in enumerate(class_names) if i > 0}} for g in ('clean_atk', 'pgd', 'clean_wb', 'wb')]
T['attack_recall'] = pd.DataFrame(rows); print(T['attack_recall'].to_string(index=False))

print('\n7c. Can the conformal score flag adversarial rows? AUROC (0.5 = no signal; below 0.5 = attacked rows look MORE normal than the reference).')
print('    A = clean attack rows vs the same rows after the attack.  B = genuine benign rows vs the attacked rows that actually evaded (arg-max benign) -- the case that matters operationally.')
rows = []
for g in ('pgd', 'wb'):
    for sc in ('1-pmax', 'setsize'):
        rows.append({'attack': g, 'signal': sc, 'AUROC A (clean vs attacked)': ms(V(f'det|{g}|A|{sc}')), 'AUROC B (benign vs evaded)': ms(V(f'det|{g}|B|{sc}')), 'evaded rows (mean)': f'{V(f"det|{g}|n_evaded").mean():.0f}'})
T['detection'] = pd.DataFrame(rows); print(T['detection'].to_string(index=False))

## 8. Supplementary statistics: clean F1 with burst-aware intervals, calibration

In [ ]:
print('8a. Clean test F1 of the full-train model (NB2 setting): seed mean +/- std, and 95 % bootstrap intervals. burst = resample bursts (the honest one); row = naive resample of rows. Intervals are the mean of the 5 seeds'' bounds.')
rows = []
for nm in ('weighted', 'macro'):
    rows.append({'metric': f'{nm} F1', 'seed mean +/- std': ms(V(f'perf|full|{nm}'), 4), 'burst CI 95%': f'[{np.mean([RES[s][f"ci|burst|{nm}"][0] for s in SEEDS]):.4f}, {np.mean([RES[s][f"ci|burst|{nm}"][1] for s in SEEDS]):.4f}]', 'row CI 95% (naive)': f'[{np.mean([RES[s][f"ci|row|{nm}"][0] for s in SEEDS]):.4f}, {np.mean([RES[s][f"ci|row|{nm}"][1] for s in SEEDS]):.4f}]'})
for i, c in enumerate(class_names):
    for nm in ('f1_pc', 'recall_pc'):
        lo_b = np.mean([RES[s][f'ci|burst|{nm}'][0][i] for s in SEEDS]); hi_b = np.mean([RES[s][f'ci|burst|{nm}'][1][i] for s in SEEDS]); lo_r = np.mean([RES[s][f'ci|row|{nm}'][0][i] for s in SEEDS]); hi_r = np.mean([RES[s][f'ci|row|{nm}'][1][i] for s in SEEDS])
        rows.append({'metric': f'{c} {"F1" if nm == "f1_pc" else "recall"}', 'seed mean +/- std': ms(V(f'perf|full|{nm}')[:, i], 4), 'burst CI 95%': f'[{lo_b:.4f}, {hi_b:.4f}]', 'row CI 95% (naive)': f'[{lo_r:.4f}, {hi_r:.4f}]'})
T['clean_f1_ci'] = pd.DataFrame(rows); print(T['clean_f1_ci'].to_string(index=False))
print(f'   The seed std only captures model variance (training randomness); the burst interval captures the finite number of independent test bursts ({nb_te}). Compare with the NB2 headline 0.8635 +/- 0.0082.')
print(f'   conformal model (proper-train, 85 % of the series) weighted F1 {ms(V("perf|cm|weighted"), 4)} vs full-train {ms(V("perf|full|weighted"), 4)}: paired p = {fp(ptest(V("perf|cm|weighted"), V("perf|full|weighted")))}, dz {dz(V("perf|cm|weighted"), V("perf|full|weighted")):.2f}')

print('\n8b. Calibration of the predicted probabilities on the test split (mean +/- std over seeds). cm_T = conformal model after temperature scaling fitted on the calibration set.')
rows = []
for nm, lab in [('cm', 'conformal model'), ('cm_T', 'conformal model + temperature'), ('full', 'full-train model')]:
    rows.append({'model': lab, 'ECE (15 bins)': ms(V(f'calib|{nm}|ece'), 4), 'Brier (multiclass)': ms(V(f'calib|{nm}|brier'), 4), 'NLL': ms(V(f'calib|{nm}|nll'), 4)})
T['calibration'] = pd.DataFrame(rows); print(T['calibration'].to_string(index=False))
print(f'   fitted temperature T = {ms(V("calib|T"), 3)} | ECE on the calibration set itself {ms(V("calib|cal_set|ece"), 4)} | temperature scaling vs raw: ECE p = {fp(ptest(V("calib|cm_T|ece"), V("calib|cm|ece")))} (dz {dz(V("calib|cm_T|ece"), V("calib|cm|ece")):.2f}), NLL p = {fp(ptest(V("calib|cm_T|nll"), V("calib|cm|nll")))}')
rb = pd.DataFrame(RES[SEEDS[0]]['calib|cm|bins'], columns=['bin', 'rows', 'mean confidence', 'accuracy']); T['reliability'] = rb; print('\n8c. Reliability bins (conformal model, seed %d)' % SEEDS[0]); print(rb.to_string(index=False))
T['burst_structure'] = pd.DataFrame({'class': class_names, 'calibration bursts': bc_cal, 'test bursts': bc_te, 'test rows': np.bincount(y_test, minlength=6), 'calibration rows': np.bincount(y_cal, minlength=6)})
print('\n8d. Independent units'); print(T['burst_structure'].to_string(index=False))

## 9. Figures and saved outputs

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 10)); col = {'lac': 'C0', 'lac_cc': 'C1', 'aps': 'C2', 'lac_dedup': 'C3'}
nom = np.array([1 - a for a in ALPHAS])
for m in METHODS: ax[0, 0].errorbar(nom, [V(f'clean|{m}|{a:g}|cov').mean() for a in ALPHAS], yerr=[V(f'clean|{m}|{a:g}|cov').std(ddof=1) for a in ALPHAS], marker='o', capsize=3, label=m, color=col[m])
ax[0, 0].plot([0.98, 1], [0.98, 1], 'k--', lw=1, label='nominal'); ax[0, 0].set_xlabel('nominal coverage 1 - alpha'); ax[0, 0].set_ylabel('test coverage'); ax[0, 0].set_title('Clean test coverage (temporal split)'); ax[0, 0].legend(fontsize=8); ax[0, 0].grid(alpha=.3)
for m in METHODS: ax[0, 1].errorbar(nom, [V(f'clean|{m}|{a:g}|size').mean() for a in ALPHAS], yerr=[V(f'clean|{m}|{a:g}|size').std(ddof=1) for a in ALPHAS], marker='o', capsize=3, label=m, color=col[m])
ax[0, 1].set_xlabel('nominal coverage 1 - alpha'); ax[0, 1].set_ylabel('mean set size'); ax[0, 1].set_title('Mean prediction-set size'); ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=.3)
x6 = np.arange(6)
for off, m in enumerate(('lac', 'lac_cc', 'aps')): ax[0, 2].bar(x6 + (off - 1) * 0.27, [V(f'clean|{m}|0.1|cov_pc')[:, i].mean() for i in range(6)], 0.27, yerr=[V(f'clean|{m}|0.1|cov_pc')[:, i].std(ddof=1) for i in range(6)], capsize=2, label=m, color=col[m])
ax[0, 2].axhline(0.9, color='k', ls='--', lw=1); ax[0, 2].set_xticks(x6); ax[0, 2].set_xticklabels(class_names, fontsize=8); ax[0, 2].set_ylim(0, 1.05); ax[0, 2].set_title('Per-class coverage at alpha = 0.1'); ax[0, 2].legend(fontsize=8); ax[0, 2].grid(alpha=.3, axis='y')
grp = ['clean_atk', 'pgd', 'clean_wb', 'wb']; xs = np.arange(len(grp))
ax[1, 0].bar(xs - 0.2, [V(f'atk|{g}|lac|0.1|cov').mean() for g in grp], 0.4, yerr=[V(f'atk|{g}|lac|0.1|cov').std(ddof=1) for g in grp], capsize=2, label='coverage'); ax[1, 0].bar(xs + 0.2, [V(f'atk|{g}|lac|0.1|only_benign').mean() for g in grp], 0.4, yerr=[V(f'atk|{g}|lac|0.1|only_benign').std(ddof=1) for g in grp], capsize=2, label='set = {benign} only')
ax[1, 0].axhline(0.9, color='k', ls='--', lw=1); ax[1, 0].set_xticks(xs); ax[1, 0].set_xticklabels(grp, fontsize=8); ax[1, 0].set_title('LAC alpha = 0.1 on attack rows'); ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=.3, axis='y')
dl = [('pgd', 'A'), ('pgd', 'B'), ('wb', 'A'), ('wb', 'B')]
for off, sc in enumerate(('1-pmax', 'setsize')): ax[1, 1].bar(np.arange(4) + (off - 0.5) * 0.4, [np.nanmean(V(f'det|{g}|{k}|{sc}')) for g, k in dl], 0.4, label=sc)
ax[1, 1].axhline(0.5, color='k', ls='--', lw=1); ax[1, 1].set_xticks(np.arange(4)); ax[1, 1].set_xticklabels([f'{g} {k}' for g, k in dl], fontsize=8); ax[1, 1].set_ylim(0, 1.05); ax[1, 1].set_title('AUROC of the conformal score as adversarial signal'); ax[1, 1].legend(fontsize=8); ax[1, 1].grid(alpha=.3, axis='y')
rb = RES[SEEDS[0]]['calib|cm|bins']; ax[1, 2].plot([0, 1], [0, 1], 'k--', lw=1); ax[1, 2].scatter([r_[2] for r_ in rb], [r_[3] for r_ in rb], s=[8 + 60 * np.log10(r_[1] + 1) for r_ in rb]); ax[1, 2].set_xlabel('mean confidence'); ax[1, 2].set_ylabel('accuracy'); ax[1, 2].set_title(f'Reliability, conformal model (seed {SEEDS[0]})'); ax[1, 2].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'conformal_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'per_seed': {s: RES[s] for s in SEEDS}, 'config': {'ALPHAS': ALPHAS, 'CAL_FRAC': CAL_FRAC, 'GAP_FRAC': GAP_FRAC, 'N_BOOT': N_BOOT, 'WB_PER_CLASS': WB_PER_CLASS, 'WB_BUDGET': WB_BUDGET, 'PGD_EPS': PGD_EPS}}, f'conformal_results_{PIPELINE_VERSION}.pkl')
for k_, t_ in T.items(): t_.to_csv(f'conformal_{k_}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB11', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb11_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'conformal_results_{PIPELINE_VERSION}.pkl', f'conformal_{PIPELINE_VERSION}.png', f'nb11_run_manifest_{PIPELINE_VERSION}.json'] + [f'conformal_{k_}_{PIPELINE_VERSION}.csv' for k_ in T]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)